In [1]:
!pip install -q -U transformers accelerate requests

import os
import json
import glob
import time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import spearmanr
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 28.7 MB/s eta 0:00:00
device: cuda


In [2]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p_ in model.parameters():
    p_.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head
layers = model.model.language_model.layers

n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = hidden_dim // n_head
IMAGE_TOKEN_ID = getattr(model.config, "image_token_id", None) or model.config.image_token_index
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim, image token id", IMAGE_TOKEN_ID)

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

32 language model layers, 32 heads, 128 head dim, image token id 32000


In [3]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
img_dir = "/kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"

YES_ID, NO_ID = 3869, 1939
NOISE_MULT = 3.0
TAIL_N = 4
GRAD_SCALE = 1.0
GROUPS = ["visual", "text", "tail"]
N_ACC = 20
N_CAL = 10
N_HEAD_TOP = 30
N_HEAD_RAND = 30
N_SEED_Q = 4
MIN_TE = 1.0
ALLOWED_SOURCES = {"amber", "pope"}


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


def resolve_image_path(q):
    qid = q["question_id"]
    if qid.startswith("pope_"):
        return os.path.join(img_dir, f"{q['image_id']}.jpg")
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    return None


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}
assert [processor.tokenizer.decode([i]).strip() for i in (YES_ID, NO_ID)] == ["Yes", "No"], processor.tokenizer.convert_ids_to_tokens([YES_ID, NO_ID])

with open(f"{INPUT}/atp-llava-selection/selection.json") as f:
    selection = json.load(f)
all_sel = [qid for t in selection["selected"] for qid in selection["selected"][t]]
print(len(all_sel), "selected questions")

rng = np.random.default_rng(0)


def pick(stratum, n, types):
    pool = [
        q for q in all_sel
        if selection["stratum"][q] == stratum and selection["source"][q] in ALLOWED_SOURCES
        and question_by_id[q]["hallucination_type"] in types
    ]
    return [str(x) for x in rng.choice(pool, size=n, replace=False)]


ALL_TYPES = ["attribute", "relation", "object"]
acc_set = pick("true_yes", N_ACC, ALL_TYPES) + pick("true_no", N_ACC, ALL_TYPES)
cal_set = (
    pick("false_yes", N_CAL - 6, ["attribute"]) + pick("false_yes", 2, ["relation"]) + pick("false_yes", 2, ["object"])
    + pick("true_yes", 1, ["attribute"]) + pick("true_no", 1, ["attribute"])
)[:N_CAL]
seed_set = cal_set[:N_SEED_Q]
all_check = list(dict.fromkeys(acc_set + cal_set))
print(len(acc_set), "accuracy questions,", len(cal_set), "calibration questions")

dla_ld = {}
for path in glob.glob(f"{INPUT}/dla-llava/*_batch_*.pt"):
    for e in torch.load(path):
        if e["question_id"] in all_check:
            dla_ld[e["question_id"]] = float(e["logit_diff"])
assert set(dla_ld) == set(all_check), "dla-llava is missing some checked questions"


def prep(qid):
    q = question_by_id[qid]
    inputs = build_inputs(load_image(resolve_image_path(q)), q["question_text"])
    ids = inputs["input_ids"]
    T = ids.shape[1]
    vis = ids[0] == IMAGE_TOKEN_ID
    tail = torch.zeros(T, dtype=torch.bool, device=ids.device)
    tail[-TAIL_N:] = True
    text = ~vis & ~tail
    assert int(vis.sum()) == 576 and not bool((vis & tail).any())
    return {
        "qid": qid, "T": T, "input_ids": ids, "pixel_values": inputs["pixel_values"],
        "attention_mask": inputs["attention_mask"], "vis": vis,
        "masks": {"visual": vis.float().cpu(), "text": text.float().cpu(), "tail": tail.float().cpu()},
        "sign": 1.0 if q["ground_truth_answer"].strip().lower() == "yes" else -1.0,
    }


prepped = {qid: prep(qid) for qid in all_check}
p0 = prepped[cal_set[0]]
print("prompt tail:", processor.tokenizer.convert_ids_to_tokens(p0["input_ids"][0, -TAIL_N - 3:].tolist()))
print("positions:", {g: int(p0["masks"][g].sum()) for g in GROUPS}, "of", p0["T"])

8403 selected questions
40 accuracy questions, 10 calibration questions
prompt tail: ['▁image', '?', '▁A', 'SS', 'IST', 'ANT', ':']
positions: {'visual': 576, 'text': 17, 'tail': 4} of 597


In [4]:
STATE = {"probe": None, "noise": None, "seed": 0, "capture": False, "grad": False, "live": None, "vis": None, "patch": None, "final_h": None}


def _layer_pre(l):
    def hook(module, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        changed = False
        if l == 0:
            if STATE["noise"] is not None:
                vis = STATE["vis"]
                h = h.clone()
                x = h[:, vis].float()
                generator = torch.Generator().manual_seed(STATE["seed"])
                noise = torch.randn(x.shape, generator=generator).to(h.device)
                noisy = x + noise * x.std() * STATE["noise"]
                h[:, vis] = noisy.to(h.dtype)
                changed = True
                if STATE["probe"] is not None:
                    STATE["probe"]["embed"] = (float(x.std()), float(x.abs().max()), float(noisy.abs().max()), float(h.float().abs().max()))
            if STATE["grad"]:
                h = h.detach().requires_grad_(True)
                changed = True
        if STATE["probe"] is not None:
            STATE["probe"].setdefault("layers", []).append(float(h.float().abs().max()))
        if STATE["capture"]:
            STATE["live"]["rp"][l] = h
        if not changed:
            return None
        if args:
            return (h,) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs)
        kwargs["hidden_states"] = h
        return args, kwargs
    return hook


def _z_pre(l):
    def hook(module, args):
        z = args[0]
        patch = STATE["patch"]
        if patch is not None and patch["kind"] == "head" and patch["layer"] == l:
            T = z.shape[1]
            z4 = z.clone().view(1, T, n_head, head_dim)
            src = patch["src"].view(T, n_head, head_dim)
            idx = patch["idx"].to(z.device)
            z4[:, idx, patch["head"], :] = src[idx.to(src.device), patch["head"], :].to(z.device, z.dtype)
            z = z4.view(1, T, n_head * head_dim)
        if STATE["capture"]:
            STATE["live"]["z"][l] = z
        return (z,) + tuple(args[1:])
    return hook


def _mlp_fwd(l):
    def hook(module, args, output):
        out = output
        patch = STATE["patch"]
        if patch is not None and patch["kind"] == "mlp" and patch["layer"] == l:
            out = out.clone()
            idx = patch["idx"].to(out.device)
            out[:, idx] = patch["src"][idx.to(patch["src"].device)].to(out.device, out.dtype).unsqueeze(0)
        if STATE["capture"]:
            STATE["live"]["mlp"][l] = out
        return out
    return hook


def _final_norm_pre(module, args):
    if STATE["capture"]:
        STATE["final_h"] = args[0][0, -1].detach().float()


_handles = []
for l, layer in enumerate(layers):
    _handles.append(layer.register_forward_pre_hook(_layer_pre(l), with_kwargs=True))
    _handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_z_pre(l)))
    _handles.append(layer.mlp.register_forward_hook(_mlp_fwd(l)))
_handles.append(ln_f.register_forward_pre_hook(_final_norm_pre))
print(len(_handles), "hooks registered")

97 hooks registered


In [5]:
def mask_on(p, gname, dev):
    cache = p.setdefault("_mask", {})
    key = (gname, str(dev))
    if key not in cache:
        cache[key] = p["masks"][gname].to(dev)
    return cache[key]


def group_idx(p, gname):
    if gname == "all":
        return torch.arange(p["T"])
    return torch.nonzero(p["masks"][gname] > 0)[:, 0]


def make_patch(kind, layer, idx, src, head=None):
    return {"kind": kind, "layer": layer, "idx": idx, "src": src, "head": head}


def capture_run(p, noise, seed):
    STATE.update(noise=noise, seed=seed, capture=True, grad=True, vis=p["vis"], patch=None, final_h=None)
    STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
    with torch.enable_grad():
        out = model(
            input_ids=p["input_ids"], pixel_values=p["pixel_values"], attention_mask=p["attention_mask"], use_cache=False,
        )
        logits = out.logits[0, -1].float()
        delta = p["sign"] * (logits[YES_ID] - logits[NO_ID])
        live = STATE["live"]
        grads = torch.autograd.grad(delta * GRAD_SCALE, live["z"] + live["mlp"] + live["rp"])
    acts = {k: [t.detach() for t in live[k]] for k in ("z", "mlp", "rp")}
    G = {"z": list(grads[:n_layer]), "mlp": list(grads[n_layer:2 * n_layer]), "rp": list(grads[2 * n_layer:])}
    result = {"acts": acts, "grads": G, "delta": float(delta.detach()), "final_h": STATE["final_h"]}
    STATE.update(capture=False, grad=False, live=None, noise=None, final_h=None)
    del out, logits, delta, live, grads
    return result


@torch.no_grad()
def delta_with(p, noise, seed, patch=None):
    STATE.update(noise=noise, seed=seed, capture=False, grad=False, vis=p["vis"], patch=patch)
    out = model(
        input_ids=p["input_ids"], pixel_values=p["pixel_values"], attention_mask=p["attention_mask"], use_cache=False,
    )
    logits = out.logits[0, -1].float()
    STATE.update(noise=None, patch=None)
    return float(p["sign"] * (logits[YES_ID] - logits[NO_ID]))


def atp_effects(A, B, G, p):
    head = torch.zeros(len(GROUPS), n_layer, n_head)
    mlp = torch.zeros(len(GROUPS), n_layer)
    rp = torch.zeros(len(GROUPS), n_layer)
    T = p["T"]
    for l in range(n_layer):
        dz = ((A["z"][l].float() - B["z"][l].float()) * G["z"][l].float() / GRAD_SCALE)[0].view(T, n_head, head_dim).sum(-1)
        dm = ((A["mlp"][l].float() - B["mlp"][l].float()) * G["mlp"][l].float() / GRAD_SCALE)[0].sum(-1)
        dr = ((A["rp"][l].float() - B["rp"][l].float()) * G["rp"][l].float() / GRAD_SCALE)[0].sum(-1)
        for gi, g in enumerate(GROUPS):
            head[gi, l] = (mask_on(p, g, dz.device) @ dz).cpu()
            mlp[gi, l] = (mask_on(p, g, dm.device) @ dm).cpu()
            rp[gi, l] = (mask_on(p, g, dr.device) @ dr).cpu()
    return {"head": head, "mlp": mlp, "rp": rp}

In [6]:
t0 = time.time()
clean_delta = {qid: delta_with(prepped[qid], None, 0) for qid in all_check}
print(f"{len(all_check)} clean forwards in {time.time() - t0:.1f} s ({(time.time() - t0) / len(all_check):.2f} s each)")

err = {qid: abs(prepped[qid]["sign"] * clean_delta[qid] - dla_ld[qid]) for qid in all_check}
print(f"clean logit difference vs dla-llava: max err {max(err.values()):.4f}, mean err {np.mean(list(err.values())):.4f}")
assert all(np.isfinite(list(clean_delta.values()))), "non-finite clean deltas"
assert max(err.values()) < 0.25, "prompt or readout differs from the DLA collection"

p_dbg = prepped[cal_set[0]]
for mult in (1.0, 2.0, 3.0, 5.0):
    STATE["probe"] = {}
    d_dbg = delta_with(p_dbg, mult, 0)
    probe, STATE["probe"] = STATE["probe"], None
    std_, max_in, max_noisy, max_h = probe["embed"]
    lm = probe["layers"]
    bad = next((i for i, v in enumerate(lm) if not np.isfinite(v)), None)
    print(f"noise x{mult}: visual embedding std {std_:.3f} max|x| {max_in:.1f} -> after noise max|x| {max_noisy:.1f} (fp16 limit 65504) | delta {d_dbg:+.2f} | "
          f"max|residual| at layers 1, 2, 4, 8, 16, 31: {[round(lm[i], 1) for i in (1, 2, 4, 8, 16, 31) if i < len(lm)]} | first non-finite layer input: {bad}")

CANDIDATES = [1.0, 1.5, 2.0, 3.0, 5.0]
noise_acc, noise_finite = {None: float(np.mean(np.array([clean_delta[q] for q in acc_set]) > 0))}, {}
print(f"clean: accuracy {noise_acc[None]:.2f} on {len(acc_set)} balanced true_yes and true_no questions")
for mult in CANDIDATES:
    deltas = np.array([delta_with(prepped[qid], mult, 0) for qid in acc_set])
    noise_finite[mult] = bool(np.isfinite(deltas).all())
    noise_acc[mult] = float(np.mean(deltas > 0)) if noise_finite[mult] else float("nan")
    print(f"noise x{mult}: finite {noise_finite[mult]}, accuracy {noise_acc[mult]:.2f}, mean delta {np.nanmean(deltas):+.2f}, non-finite runs {int((~np.isfinite(deltas)).sum())}/{len(deltas)}")
ok = [m for m in CANDIDATES if noise_finite[m] and 0.3 <= noise_acc[m] <= 0.7]
assert ok and noise_acc[None] > 0.9, "no usable noise level: see the per-layer residual maxima above"
NOISE_MULT = max(ok)
print(f"using NOISE_MULT = {NOISE_MULT} (the largest finite multiplier with near-chance accuracy; the paper's 3 sigma is {'kept' if NOISE_MULT == 3.0 else 'not reachable in fp16 here'})")


50 clean forwards in 22.8 s (0.46 s each)
clean logit difference vs dla-llava: max err 0.0000, mean err 0.0000
noise x1.0: visual embedding std 0.992 max|x| 35.8 -> after noise max|x| 36.3 (fp16 limit 65504) | delta -0.66 | max|residual| at layers 1, 2, 4, 8, 16, 31: [36.3, 1332.0, 1336.0, 1344.0, 1344.0, 804.5] | first non-finite layer input: None
noise x2.0: visual embedding std 0.992 max|x| 35.8 -> after noise max|x| 37.3 (fp16 limit 65504) | delta -1.32 | max|residual| at layers 1, 2, 4, 8, 16, 31: [37.3, 1332.0, 1336.0, 1344.0, 1344.0, 16160.0] | first non-finite layer input: None
noise x3.0: visual embedding std 0.992 max|x| 35.8 -> after noise max|x| 38.3 (fp16 limit 65504) | delta +nan | max|residual| at layers 1, 2, 4, 8, 16, 31: [38.3, 1332.0, 1336.0, 1344.0, 1344.0, nan] | first non-finite layer input: 31
noise x5.0: visual embedding std 0.992 max|x| 35.8 -> after noise max|x| 40.3 (fp16 limit 65504) | delta +nan | max|residual| at layers 1, 2, 4, 8, 16, 31: [40.3, 1332.0, 1

/tmp/ipykernel_23/1769257361.py:28: RuntimeWarning: Mean of empty slice
  print(f"noise x{mult}: finite {noise_finite[mult]}, accuracy {noise_acc[mult]:.2f}, mean delta {np.nanmean(deltas):+.2f}, non-finite runs {int((~np.isfinite(deltas)).sum())}/{len(deltas)}")


noise x3.0: finite False, accuracy nan, mean delta +nan, non-finite runs 40/40
noise x5.0: finite False, accuracy nan, mean delta +nan, non-finite runs 40/40
using NOISE_MULT = 1.5 (the largest finite multiplier with near-chance accuracy; the paper's 3 sigma is not reachable in fp16 here)


In [7]:
SEED0 = 1000
cal_results = []
timing = {"capture": [], "exact": []}
u_vec = ln_f.weight.float() * (lm_head.weight[YES_ID] - lm_head.weight[NO_ID]).float()

for qi, qid in enumerate(cal_set):
    p = prepped[qid]
    seed = SEED0 + qi
    t0 = time.time()
    c = capture_run(p, None, seed)
    r = capture_run(p, NOISE_MULT, seed)
    timing["capture"].append((time.time() - t0) / 2)
    assert np.isfinite(r["delta"]) and np.isfinite(c["delta"]), f"non-finite delta for {qid}"
    TE = c["delta"] - r["delta"]
    atp = {"den": atp_effects(c["acts"], r["acts"], r["grads"], p), "noi": atp_effects(r["acts"], c["acts"], c["grads"], p)}
    selfcheck = abs(delta_with(p, NOISE_MULT, seed) - r["delta"])

    anchors = {}
    for direction, src, base, base_run in (("den", c, r, r), ("noi", r, c, c)):
        h = base_run["final_h"]
        rms = torch.sqrt(h.pow(2).mean() + ln_f.variance_epsilon)
        dm = (src["acts"]["mlp"][-1][0, -1].float() - base["acts"]["mlp"][-1][0, -1].float()).to(h.device)
        anchors[direction] = float(p["sign"] * (dm @ u_vec.to(h.device)) / rms)

    exact = {"den": {"mlp": {}, "head": {}}, "noi": {"mlp": {}, "head": {}}}
    t1 = time.time()
    n_fwd = 0
    for direction in ("den", "noi"):
        base_noise = NOISE_MULT if direction == "den" else None
        src = c["acts"] if direction == "den" else r["acts"]
        base_delta = r["delta"] if direction == "den" else c["delta"]
        head_all = atp[direction]["head"].sum(0).flatten()
        top = torch.topk(head_all.abs(), N_HEAD_TOP).indices.tolist()
        rest = np.setdiff1d(np.arange(head_all.numel()), top)
        rand = np.random.default_rng(qi).choice(rest, size=N_HEAD_RAND, replace=False).tolist()
        for l in range(n_layer):
            for gname in ["all"] + GROUPS:
                patch = make_patch("mlp", l, group_idx(p, gname), src["mlp"][l][0])
                exact[direction]["mlp"][(gname, l)] = delta_with(p, base_noise, seed, patch) - base_delta
                n_fwd += 1
        for flat in top + rand:
            l, h_ = divmod(int(flat), n_head)
            patch = make_patch("head", l, group_idx(p, "all"), src["z"][l][0], h_)
            exact[direction]["head"][(l, h_)] = delta_with(p, base_noise, seed, patch) - base_delta
            n_fwd += 1
    timing["exact"].append((time.time() - t1) / n_fwd)

    cal_results.append({
        "qid": qid, "stratum": selection["stratum"][qid], "TE": TE, "delta_clean": c["delta"], "delta_corrupt": r["delta"],
        "selfcheck": selfcheck, "anchors": anchors, "atp": atp, "exact": exact,
    })
    print(f"[{qi + 1}/{len(cal_set)}] {qid} {selection['stratum'][qid]}: clean {c['delta']:+.2f} corrupt {r['delta']:+.2f} TE {TE:+.2f} | "
          f"grad run {timing['capture'][-1]:.1f} s, exact {timing['exact'][-1]:.2f} s/forward")
    del c, r, atp
    torch.cuda.empty_cache()

[1/10] amber_2002 false_yes: clean -0.69 corrupt -0.73 TE +0.05 | grad run 1.0 s, exact 0.50 s/forward
[2/10] amber_2552 false_yes: clean -3.92 corrupt -0.52 TE -3.40 | grad run 1.0 s, exact 0.53 s/forward
[3/10] amber_5076 false_yes: clean -4.14 corrupt -0.37 TE -3.77 | grad run 1.1 s, exact 0.53 s/forward
[4/10] amber_7460 false_yes: clean -1.72 corrupt +0.12 TE -1.84 | grad run 1.1 s, exact 0.54 s/forward
[5/10] amber_13563 false_yes: clean -0.09 corrupt +0.01 TE -0.10 | grad run 1.1 s, exact 0.54 s/forward
[6/10] amber_14791 false_yes: clean -0.11 corrupt +0.15 TE -0.26 | grad run 1.1 s, exact 0.54 s/forward
[7/10] pope_2663 false_yes: clean -0.88 corrupt -0.49 TE -0.38 | grad run 1.1 s, exact 0.57 s/forward
[8/10] pope_2163 false_yes: clean -2.89 corrupt -0.59 TE -2.30 | grad run 1.2 s, exact 0.57 s/forward
[9/10] amber_3991 true_yes: clean +1.23 corrupt -0.09 TE +1.32 | grad run 1.2 s, exact 0.59 s/forward
[10/10] amber_1666 true_no: clean +1.44 corrupt -0.20 TE +1.63 | grad run 

In [8]:
def pair_frame(direction):
    rows = []
    for res in cal_results:
        a, e = res["atp"][direction], res["exact"][direction]
        mlp_all, head_all = a["mlp"].sum(0), a["head"].sum(0)
        for (g, l), v in e["mlp"].items():
            atp_v = mlp_all[l].item() if g == "all" else a["mlp"][GROUPS.index(g), l].item()
            rows.append({"qid": res["qid"], "kind": f"mlp_{g}", "comp": l, "atp": atp_v, "exact": v})
        for (l, h_), v in e["head"].items():
            rows.append({"qid": res["qid"], "kind": "head_all", "comp": l * n_head + h_, "atp": head_all[l, h_].item(), "exact": v})
    return pd.DataFrame(rows)


def calib_stats(df):
    out = {
        "pooled_spearman": float(spearmanr(df.atp, df.exact)[0]),
        "slope_exact_on_atp": float((df.atp * df.exact).sum() / (df.atp ** 2).sum()),
    }
    for kind, sub in df.groupby("kind"):
        out[f"spearman_{kind}"] = float(spearmanr(sub.atp, sub.exact)[0])
    key = df[df.kind.isin(["mlp_all", "head_all"])]
    rec, sgn, perq = [], [], []
    for qid, sub in key.groupby("qid"):
        sub = sub.reset_index(drop=True)
        k = min(10, len(sub))
        rec.append(len(set(sub.exact.abs().nlargest(k).index) & set(sub.atp.abs().nlargest(k).index)) / k)
        big = sub[sub.exact.abs() > 0.1 * sub.exact.abs().max()]
        sgn.append(float((np.sign(big.atp) == np.sign(big.exact)).mean()))
        perq.append(float(spearmanr(sub.atp, sub.exact)[0]))
    out.update(top10_recall=float(np.mean(rec)), sign_agreement=float(np.mean(sgn)), per_question_spearman=float(np.mean(perq)))
    return out


calib = {}
for direction, name in (("den", "denoising"), ("noi", "noising")):
    df = pair_frame(direction)
    calib[direction] = calib_stats(df)
    s = calib[direction]
    print(f"--- {name}: {len(df)} (question, component) pairs ---")
    print({k: round(v, 3) for k, v in s.items()})
    meets = s["pooled_spearman"] >= 0.8 and s["top10_recall"] >= 0.7
    print("meets the bars (pooled spearman >= 0.8, top-10 recall >= 0.7):", meets)

mid = min(n_layer // 2 + 2, n_layer - 1)
print(f"--- L{mid} MLP, all positions: atp vs exact per question ---")
for res in cal_results:
    print(f"{res['qid']:14s} {res['stratum']:10s} TE {res['TE']:+7.2f} | den atp {res['atp']['den']['mlp'].sum(0)[mid]:+7.3f} exact {res['exact']['den']['mlp'][('all', mid)]:+7.3f}"
          f" | noi atp {res['atp']['noi']['mlp'].sum(0)[mid]:+7.3f} exact {res['exact']['noi']['mlp'][('all', mid)]:+7.3f}")

print("--- residual stream identity: exact effect of patching resid_pre at all positions equals TE (denoising) or minus TE (noising) ---")
layers_rp = sorted({0, n_layer // 4, n_layer // 2, 3 * n_layer // 4, n_layer - 1})
use = [res for res in cal_results if abs(res["TE"]) > MIN_TE]
for l in layers_rp:
    den_ratio = [res["atp"]["den"]["rp"].sum(0)[l].item() / res["TE"] for res in use]
    noi_ratio = [res["atp"]["noi"]["rp"].sum(0)[l].item() / -res["TE"] for res in use]
    print(f"resid_pre L{l:2d}: atp / exact  denoising median {np.median(den_ratio) if use else float('nan'):.2f}"
          f" | noising median {np.median(noi_ratio) if use else float('nan'):.2f}  ({len(use)} questions with |TE| > {MIN_TE})")

print("--- anchor: last-layer MLP, atp vs the DLA-style frozen-norm prediction ---")
for res in cal_results:
    print(f"{res['qid']:14s} den atp {res['atp']['den']['mlp'].sum(0)[-1]:+8.3f} dla {res['anchors']['den']:+8.3f}"
          f" | noi atp {res['atp']['noi']['mlp'].sum(0)[-1]:+8.3f} dla {res['anchors']['noi']:+8.3f}")
print(f"grad run vs no-grad run delta, max abs diff: {max(r_['selfcheck'] for r_ in cal_results):.4f}")

--- denoising: 1880 (question, component) pairs ---
{'pooled_spearman': 0.772, 'slope_exact_on_atp': 0.454, 'spearman_head_all': 0.915, 'spearman_mlp_all': 0.752, 'spearman_mlp_tail': 0.746, 'spearman_mlp_text': 0.536, 'spearman_mlp_visual': 0.879, 'top10_recall': 0.52, 'sign_agreement': 0.939, 'per_question_spearman': 0.834}
meets the bars (pooled spearman >= 0.8, top-10 recall >= 0.7): False
--- noising: 1880 (question, component) pairs ---
{'pooled_spearman': 0.78, 'slope_exact_on_atp': 0.814, 'spearman_head_all': 0.93, 'spearman_mlp_all': 0.712, 'spearman_mlp_tail': 0.849, 'spearman_mlp_text': 0.532, 'spearman_mlp_visual': 0.773, 'top10_recall': 0.63, 'sign_agreement': 0.921, 'per_question_spearman': 0.824}
meets the bars (pooled spearman >= 0.8, top-10 recall >= 0.7): False
--- L18 MLP, all positions: atp vs exact per question ---
amber_2002     false_yes  TE   +0.05 | den atp  +0.739 exact  +0.648 | noi atp  -0.419 exact  -0.406
amber_2552     false_yes  TE   -3.40 | den atp  -0.

In [9]:
vec = lambda a: torch.cat([a["head"].sum(0).flatten(), a["mlp"].sum(0)]).numpy()

seed_stab = {"den": [], "noi": []}
te_spread = []
for qid in seed_set:
    p = prepped[qid]
    c = capture_run(p, None, 0)
    runs = {"den": [], "noi": []}
    tes = []
    for s in range(3):
        r = capture_run(p, NOISE_MULT, 100 + s)
        runs["den"].append(vec(atp_effects(c["acts"], r["acts"], r["grads"], p)))
        runs["noi"].append(vec(atp_effects(r["acts"], c["acts"], c["grads"], p)))
        tes.append(c["delta"] - r["delta"])
        del r
    for direction in ("den", "noi"):
        v = runs[direction]
        seed_stab[direction].append(np.mean([spearmanr(v[i], v[j])[0] for i in range(3) for j in range(i + 1, 3)]))
    te_spread.append(float(np.std(tes)))
    del c
    torch.cuda.empty_cache()
for direction in ("den", "noi"):
    print(f"{direction}: rank correlation of effects across noise seeds, mean {np.mean(seed_stab[direction]):.3f} (per question {np.round(seed_stab[direction], 3).tolist()})")
print("std of TE across seeds per question:", np.round(te_spread, 3).tolist())

p = prepped[seed_set[0]]
scale_vecs = {}
for scale in (1.0, 256.0):
    GRAD_SCALE = scale
    c = capture_run(p, None, 0)
    r = capture_run(p, NOISE_MULT, 100)
    scale_vecs[scale] = vec(atp_effects(c["acts"], r["acts"], r["grads"], p))
    del c, r
    torch.cuda.empty_cache()
GRAD_SCALE = 1.0
a_, b_ = scale_vecs[1.0], scale_vecs[256.0]
assert np.isfinite(a_).all() and np.isfinite(b_).all(), "non-finite attribution effects"
big = np.abs(a_) > 0.01 * np.abs(a_).max()
grad_scale_err = float(np.max(np.abs(a_[big] - b_[big]) / np.abs(a_[big])))
print(f"grad scale 1 vs 256: max relative difference on the larger effects {grad_scale_err:.4f}, spearman {spearmanr(a_, b_)[0]:.4f}")

den: rank correlation of effects across noise seeds, mean 0.638 (per question [0.665, 0.737, 0.599, 0.55])
noi: rank correlation of effects across noise seeds, mean 0.904 (per question [0.905, 0.927, 0.896, 0.887])
std of TE across seeds per question: [0.12, 0.16, 0.215, 0.154]
grad scale 1 vs 256: max relative difference on the larger effects 0.1069, spearman 1.0000


In [10]:
summary = {
    "noise_accuracy": {str(k): v for k, v in noise_acc.items()},
    "clean_vs_dla_max_err": float(max(err.values())),
    "calibration": calib,
    "seed_stability": {d: float(np.mean(v)) for d, v in seed_stab.items()},
    "grad_scale_max_rel_diff": grad_scale_err,
    "sec_per_grad_run": float(np.mean(timing["capture"])),
    "sec_per_exact_forward": float(np.mean(timing["exact"])),
    "max_gpu_mem_gb": [torch.cuda.max_memory_allocated(i) / 1e9 for i in range(torch.cuda.device_count())],
    "noise_mult": NOISE_MULT, "tail_n": TAIL_N,
}
for key, val in summary.items():
    print(key, val)

print(f"estimated collection speed for both directions: about {2 * summary['sec_per_grad_run']:.1f} s per question, "
      f"{len(all_sel) * 2 * summary['sec_per_grad_run'] / 3600:.1f} h for all {len(all_sel)} selected questions")

with open("/kaggle/working/atp_validate_llava.json", "w") as f:
    json.dump(summary, f, indent=1)

noise_accuracy {'None': 0.975, '1.0': 0.55, '1.5': 0.475, '2.0': nan, '3.0': nan, '5.0': nan}
clean_vs_dla_max_err 0.0
calibration {'den': {'pooled_spearman': 0.7718703220132027, 'slope_exact_on_atp': 0.4539403043846069, 'spearman_head_all': 0.9152253361602463, 'spearman_mlp_all': 0.7522391178105557, 'spearman_mlp_tail': 0.7464702837428453, 'spearman_mlp_text': 0.5364317508765443, 'spearman_mlp_visual': 0.8786092708549071, 'top10_recall': 0.5199999999999999, 'sign_agreement': 0.939482509180316, 'per_question_spearman': 0.8336366940196598}, 'noi': {'pooled_spearman': 0.7798026560158323, 'slope_exact_on_atp': 0.8144307656114654, 'spearman_head_all': 0.9295421224786351, 'spearman_mlp_all': 0.7121300109673016, 'spearman_mlp_tail': 0.8491107375057242, 'spearman_mlp_text': 0.5321501283462523, 'spearman_mlp_visual': 0.772931224274982, 'top10_recall': 0.63, 'sign_agreement': 0.9210254589293753, 'per_question_spearman': 0.82376658685299}}
seed_stability {'den': 0.6375456409612587, 'noi': 0.9037